# Qdrant × Future AGI: fixing RAG decay

[Full workshop and Streamlit app](https://github.com/future-agi/cookbooks/tree/main/qdrant-rag-decay-workshop) · [Open in Colab](https://colab.research.google.com/github/future-agi/cookbooks/blob/main/qdrant-rag-decay-workshop/workshop.ipynb)

Can your retriever still find the current Steel type chart after a stale page is crawled five times? Build a small Pokédex index, measure retrieval before and after each change, and compare two generated answers with the current chart. The in-memory collection is private to this kernel; it does not change the Streamlit app or a hosted Qdrant collection. If you are presenting the original Streamlit demo, use the [paired notebook at the webinar revision](https://github.com/future-agi/cookbooks/blob/a14882ecc246aad2ffb7e0513461f95af8f814c9/qdrant-rag-decay-workshop/workshop.ipynb) with its dedicated, prepared collection. That demo uses different encoders and 37 golden queries; its scores are not reference values for this example.

## 1. Set up the index and tracing

Start in a fresh Python 3.12 kernel. You need an OpenAI key for embeddings and chat, and Future AGI credentials for traces and hosted evaluations. No Qdrant account is needed. If you self-host Future AGI, configure `FI_BASE_URL` and `FI_TRACES_URL` before registration.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "qdrant-client==1.18.0", "agent-learning-kit==0.3.7", "fi-instrumentation-otel", "openai"], check=True);

In [ ]:
import os
from getpass import getpass

for name in ('OPENAI_API_KEY', 'FI_API_KEY', 'FI_SECRET_KEY'):
    if not os.environ.get(name):
        os.environ[name] = getpass(f"{name}: ")

In [ ]:
import math
import re
from collections import Counter
from uuid import uuid4

from fi.evals import Evaluator
from fi_instrumentation import register
from fi_instrumentation.fi_types import ProjectType
from openai import OpenAI
from qdrant_client import QdrantClient, models as qm

client = QdrantClient(":memory:")
llm, evaluator = OpenAI(), Evaluator()
collection = "pokedex"
project = "pokedex-rag-workshop"
run_id = uuid4().hex[:12]
provider = register(project_type=ProjectType.OBSERVE, project_name=project, verbose=False)
tracer = provider.get_tracer("pokedex")

# The document ID identifies a version. Only identical chunks within a version are duplicates.
docs = {
    "steel-new": (True, "Steel takes normal damage from Ghost and Dark attacks in the current games."),
    "steel-old": (False, "Does the Steel type resist Ghost and Dark attacks? Yes, Steel resists both."),
    "gengar": (True, "Gengar hides in shadows. Its presence chills the surrounding air."),
    "pikachu": (True, "Pikachu is an electric mouse. It stores electricity in its cheek pouches."),
    "marowak": (True, "Marowak throws a bone like a boomerang. Its field guide code is R17."),
    "drowzee": (True, "Drowzee feeds on dreams. It is a Psychic-type Pokémon."),
    "psychic": (True, "The Psychic type is weak to Bug attacks."),
}
cases = [
    ("Does the Steel type resist Ghost and Dark attacks?", "steel-new"),
    ("Which electric mouse has cheek pouches?", "pikachu"),
    ("Which entry has field guide code R17?", "marowak"),
]
steel_question = cases[0][0]

def words(text):
    return re.findall(r"[^\W_]+", text.lower())

vocab = {word: i for i, word in enumerate(sorted({w for _, text in docs.values() for w in words(text)}))}

def sparse(text):
    counts = sorted(Counter(vocab[w] for w in words(text) if w in vocab).items())
    return qm.SparseVector(indices=[i for i, _ in counts], values=[float(v) for _, v in counts])

# Cache each model/text pair; use matching encoders at index time and query time.
cache = {}
encoders = {"small": "text-embedding-3-small", "large": "text-embedding-3-large"}

def embed(texts, encoder):
    missing = list(dict.fromkeys(t for t in texts if (encoder, t) not in cache))
    if missing:
        result = llm.embeddings.create(model=encoders[encoder], dimensions=256, input=missing)
        for row in result.data:
            cache[encoder, missing[row.index]] = row.embedding
    return [cache[encoder, t] for t in texts]

texts = [text for _, text in docs.values()]
segments = {t: [s.strip() for s in re.split(r"[.!?]", t) if s.strip()] for t in texts}
embed(texts + [q for q, _ in cases], "small");
embed(texts + [q for q, _ in cases] + [s for parts in segments.values() for s in parts], "large");

## 2. Measure decay as the corpus grows

Index the current entries first. Each point carries two named dense vectors, a lexical vector, and one vector per sentence for later reranking. Score three fixed queries. Then add five copies of an obsolete type chart and rerun the *same* queries; note which document IDs occupy the top three.

In [ ]:
dims = dict(size=256, distance=qm.Distance.COSINE)
client.create_collection(collection, vectors_config={
    "small": qm.VectorParams(**dims), "large": qm.VectorParams(**dims),
    "sentences": qm.VectorParams(**dims, multivector_config=qm.MultiVectorConfig(
        comparator=qm.MultiVectorComparator.MAX_SIM)),
}, sparse_vectors_config={"lexical": qm.SparseVectorParams()})

def point(point_id, doc_id):
    current, text = docs[doc_id]
    return qm.PointStruct(id=point_id, payload={"doc_id": doc_id, "text": text,
                          "chunk": 0, "is_current": current}, vector={
        "small": embed([text], "small")[0], "large": embed([text], "large")[0],
        "lexical": sparse(text), "sentences": embed(segments[text], "large"),
    })

client.upsert(collection, [point(i, doc_id) for i, doc_id in enumerate(docs) if docs[doc_id][0]])

def dense_search(question, encoder="small"):
    return client.query_points(collection, query=embed([question], encoder)[0], using=encoder, limit=3).points

def measure(stage, search):
    ranks = []
    for question, gold in cases:
        ids = [p.payload["doc_id"] for p in search(question)]
        rank = ids.index(gold) + 1 if gold in ids else None
        print(f"  {gold}: {ids}, gold rank={rank}")
        ranks.append(rank)
    hit_rate = sum(rank is not None for rank in ranks) / len(ranks)
    mrr = sum(1/rank for rank in ranks if rank) / len(ranks)
    ndcg = sum(1/math.log2(1 + rank) for rank in ranks if rank) / len(ranks)
    print(f"{stage}: recall@3={hit_rate:.2f}, MRR@3={mrr:.2f}, nDCG@3={ndcg:.2f}")

measure("Clean corpus", dense_search)
client.upsert(collection, [point(i, "steel-old") for i in range(100, 105)])
measure("After recrawl", dense_search)
stale_hits = dense_search(steel_question)

## 3. Deduplicate the repeated chunks

Dropping all versions by topic would discard the current chart too. Keep one copy of each `(doc_id, chunk)` pair. Only the collection created in this kernel is modified. Compare duplicate rate with relevance: five copies becoming one does not make that one copy current.

In [ ]:
def duplicate_rate(hits):
    return 1 - len({(p.payload["doc_id"], p.payload["chunk"]) for p in hits}) / len(hits)

seen, duplicates, offset = set(), [], None
while True:
    points, offset = client.scroll(collection, limit=4, offset=offset, with_vectors=False)
    for record in points:
        key = (record.payload["doc_id"], record.payload["chunk"])
        if key in seen:
            duplicates.append(record.id)
        seen.add(key)
    if offset is None:
        break
if duplicates:
    client.delete(collection, points_selector=duplicates)
print("Duplicate rate at 3:", duplicate_rate(stale_hits), "->", duplicate_rate(dense_search(steel_question)))
assert client.count(collection).count == len(docs)
measure("After deduplication", dense_search)

## 4. Compare embedding models

Switch the **named vector**, not the collection or the labels. An upgrade may not move a three-query score. Look at individual ranks before deciding whether a model is worth changing.

In [ ]:
for encoder in encoders:
    measure(encoder, lambda q: dense_search(q, encoder))

## 5. Fuse dense and lexical retrieval, then rerank

Lexical search gives a rare identifier such as `R17` another way into the candidate list. Fuse the dense and lexical rankings using reciprocal rank fusion. Rerank that shortlist by the best matching sentence in each document (Qdrant `MAX_SIM` with one query vector). This is sentence-level reranking, not a ColBERT model.

In [ ]:
current = qm.Filter(must=[qm.FieldCondition(key="is_current", match=qm.MatchValue(value=True))])

def hybrid(question, current_only=False):
    filt = current if current_only else None
    vector = embed([question], "large")[0]
    candidates = qm.Prefetch(prefetch=[
        qm.Prefetch(query=vector, using="large", limit=5, filter=filt),
        qm.Prefetch(query=sparse(question), using="lexical", limit=5, filter=filt),
    ], query=qm.FusionQuery(fusion=qm.Fusion.RRF), limit=5, filter=filt)
    return client.query_points(collection, prefetch=candidates, query=[vector], using="sentences",
                               query_filter=filt, limit=3).points

measure("Hybrid + sentence reranking", hybrid)

## 6. Filter out obsolete facts and score the answers

A stale document can be an excellent semantic match. Filter by `is_current` during candidate selection, not after generation. Generate answers from the original and filtered results. Judge each answer against the *current* chart; grading it only against the passages it saw could reward repetition of an obsolete fact.

In [ ]:
def current_search(q):
    return hybrid(q, current_only=True)

measure("Current documents only", current_search)
fresh_hits = current_search(steel_question)
assert all(hit.payload["is_current"] for hit in fresh_hits)

def answer_from(stage, question, hits):
    context = "\n".join(f"[{h.payload['doc_id']}] {h.payload['text']}" for h in hits)
    with tracer.start_as_current_span("pokedex-answer") as span:
        span.set_attribute("workshop.run", run_id)
        span.set_attribute("workshop.stage", stage)
        span.set_attribute("raw.input", question)
        span.set_attribute("retrieval.context", context)
        response = llm.chat.completions.create(model="gpt-4o-mini", temperature=0, messages=[
            {"role": "system", "content": "Answer in one sentence using only the provided excerpts. Cite a document ID. If the excerpts do not answer the question, say so."},
            {"role": "user", "content": f"{question}\n\nExcerpts:\n{context}"},
        ])
        answer = response.choices[0].message.content
        if not answer:
            raise RuntimeError("The generator returned no answer.")
        span.set_attribute("raw.output", answer)
    return answer

for stage, hits in (("recrawl", stale_hits), ("filtered", fresh_hits)):
    answer = answer_from(stage, steel_question, hits)
    result = evaluator.evaluate(eval_templates="factual_accuracy", model_name="turing_flash",
                                inputs={"input": steel_question, "output": answer,
                                        "context": docs["steel-new"][1]}).eval_results[0]
    if not result.eval_id or result.output is None:
        raise RuntimeError(f"No factual accuracy verdict for {stage}: {result.reason}")
    saved = evaluator.get_eval_result(str(result.eval_id))["result"]["result"]
    assert str(saved["eval_id"]) == str(result.eval_id) and saved["name"] == "factual_accuracy"
    print(stage, "|", answer, "| accuracy:", result.output, "|", result.reason)

## 7. Follow a second retrieval hop

First retrieve Drowzee’s entry. Read its type from the entry itself, then retrieve that type’s weakness chart. Inspect both document IDs before you accept an answer that needs evidence from both.

In [ ]:
first = current_search("What does Drowzee eat and what type is Drowzee?")[0]
match = re.search(r"(\w+)-type", first.payload["text"])
if not match:
    raise RuntimeError(f"First hop did not find a type in {first.payload['doc_id']}")
second = current_search(f"Is the {match[1]} type weak to Bug attacks?")[0]
question = "What does Drowzee eat, and is its type weak to Bug attacks?"
print("Evidence:", first.payload["doc_id"], "->", second.payload["doc_id"])
print(answer_from("two-hop", question, [first, second]))
assert provider.force_flush()
print(f"Observe project: {project}, workshop.run: {run_id}")

Open **Observe → Tracing** and find the `pokedex-answer` spans in `pokedex-rag-workshop`. Compare `workshop.stage`, `retrieval.context`, and `raw.output` for this run. Duplicate rate, a gold document’s rank, and the answer’s factual accuracy answer different questions. Carry all three into the larger workshop before drawing conclusions about a production retriever.